# Java Repository Analysis

This notebook analyzes the LOC output produced by `scripts/github_repo_java_loc_analysis.py`.

Recommended launch command from the repository root:

```bash
source .venv/bin/activate
jupyter lab
```

The thresholds are configurable in the next cell.

In [1]:
from pathlib import Path
import json
from html import escape
from IPython.display import HTML, display

DATA_PATH = Path("../java-repos3-loc.json")
MIN_TOTAL_LOC = 1500
MAX_TOTAL_LOC = 7000
ONLY_SUCCESSFUL_ROWS = True

print(f"Using analysis file: {DATA_PATH}")
print(f"LOC range: {MIN_TOTAL_LOC} to {MAX_TOTAL_LOC}")

Using analysis file: ../java-repos3-loc.json
LOC range: 1500 to 7000


In [2]:
payload = json.loads(DATA_PATH.read_text(encoding="utf-8"))
repositories = payload["repositories"] if isinstance(payload, dict) else payload

if ONLY_SUCCESSFUL_ROWS:
    repositories = [repo for repo in repositories if repo.get("status") == "ok"]

print(f"Loaded {len(repositories)} repositories")
if isinstance(payload, dict) and "summary" in payload:
    print("Existing script summary:")
    print(json.dumps(payload["summary"], indent=2))

Loaded 110 repositories
Existing script summary:
{
  "repositories_analyzed": 110,
  "successful": 110,
  "errors": 0,
  "java_main_language_count": 92,
  "java_main_language_share": 0.836364,
  "aggregate_java_loc": 603099,
  "aggregate_total_loc": 1029851,
  "aggregate_java_share": 0.585618
}


In [3]:
def as_int(value):
    return int(value or 0)


def in_loc_range(repo, minimum, maximum):
    total_loc = as_int(repo.get("java_loc"))
    return minimum <= total_loc <= maximum


repos_in_range = [repo for repo in repositories if in_loc_range(repo, MIN_TOTAL_LOC, MAX_TOTAL_LOC)]
repos_java_main = [repo for repo in repositories if repo.get("java_is_main_language")]
repos_in_range_java_main = [repo for repo in repos_in_range if repo.get("java_is_main_language")]

summary = {
    "total_repositories": len(repositories),
    "repos_with_total_loc_in_range": len(repos_in_range),
    "repos_with_java_as_main_language": len(repos_java_main),
    "repos_in_range_with_java_as_main_language": len(repos_in_range_java_main),
}

summary

{'total_repositories': 110,
 'repos_with_total_loc_in_range': 53,
 'repos_with_java_as_main_language': 92,
 'repos_in_range_with_java_as_main_language': 48}

In [4]:
def display_rows(rows, columns, limit=None):
    subset = rows if limit is None else rows[:limit]
    header_html = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body_html = []
    for row in subset:
        cells = "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns)
        body_html.append(f"<tr>{cells}</tr>")
    table_html = (
        "<table>"
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(body_html)}</tbody>"
        "</table>"
    )
    display(HTML(table_html))


columns = ["name", "total_loc", "java_loc", "java_percent", "java_is_main_language", "top_language"]

print("Repositories with total LOC inside the selected range:")
display_rows(sorted(repos_in_range, key=lambda repo: as_int(repo.get("total_loc"))), columns)


Repositories with total LOC inside the selected range:


name,total_loc,java_loc,java_percent,java_is_main_language,top_language
EsotericSoftware/reflectasm,1920,1692,88.12,True,Java
LeonardoZ/java-concurrency-patterns,1977,1884,95.3,True,Java
whwlsfb/JDumpSpider,2663,2341,87.91,True,Java
alamkanak/Android-Week-View,2808,1963,69.91,True,Java
YeautyYE/netty-websocket-spring-boot-starter,2859,2102,73.52,True,Java
JakeWharton/RxRelay,2972,2405,80.92,True,Java
esoxjem/MovieGuide,3326,2342,70.41,True,Java
monkeyWie/proxyee,3440,2755,80.09,True,Java
mbechler/marshalsec,3526,3185,90.33,True,Java
pedrovgs/EffectiveAndroidUI,3541,2225,62.84,True,Java


In [5]:
print("Repositories in range where Java is the main language:")
display_rows(
    sorted(repos_in_range_java_main, key=lambda repo: as_int(repo.get("total_loc"))),
    ["name", "total_loc", "java_loc", "java_percent", "top_language"],
)


Repositories in range where Java is the main language:


name,total_loc,java_loc,java_percent,top_language
EsotericSoftware/reflectasm,1920,1692,88.12,Java
LeonardoZ/java-concurrency-patterns,1977,1884,95.3,Java
whwlsfb/JDumpSpider,2663,2341,87.91,Java
alamkanak/Android-Week-View,2808,1963,69.91,Java
YeautyYE/netty-websocket-spring-boot-starter,2859,2102,73.52,Java
JakeWharton/RxRelay,2972,2405,80.92,Java
esoxjem/MovieGuide,3326,2342,70.41,Java
monkeyWie/proxyee,3440,2755,80.09,Java
mbechler/marshalsec,3526,3185,90.33,Java
pedrovgs/EffectiveAndroidUI,3541,2225,62.84,Java


In [6]:
import csv

export_path = DATA_PATH.with_name(
    f"{DATA_PATH.stem.replace('-loc', '')}-java-main-{MIN_TOTAL_LOC}-{MAX_TOTAL_LOC}.csv"
)

with export_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    for repo in sorted(repos_in_range_java_main, key=lambda repo: repo["name"].lower()):
        writer.writerow([repo["name"]])

print(f"Wrote {len(repos_in_range_java_main)} repository names to {export_path}")


Wrote 48 repository names to ../java-repos3-java-main-1500-7000.csv


In [7]:
BASE_COMPARE_PATH = Path("../java-repos3-loc.json")
ADD_COMPARE_PATH = Path("../java-repos-add-loc.json")

def load_repo_rows(path):
    payload = json.loads(path.read_text(encoding="utf-8"))
    repos = payload["repositories"] if isinstance(payload, dict) else payload
    return [repo for repo in repos if repo.get("status") == "ok" and "name" in repo]

base_compare_rows = load_repo_rows(BASE_COMPARE_PATH)
add_compare_rows = load_repo_rows(ADD_COMPARE_PATH)
base_repo_names = {repo["name"] for repo in base_compare_rows}
new_compare_rows = [repo for repo in add_compare_rows if repo["name"] not in base_repo_names]
new_repo_rows_in_loc_range = [
    repo for repo in new_compare_rows
    if in_loc_range(repo, MIN_TOTAL_LOC, MAX_TOTAL_LOC)
]
new_repo_names_in_loc_range = sorted(repo["name"] for repo in new_repo_rows_in_loc_range)

print(f"Base comparison file: {BASE_COMPARE_PATH}")
print(f"Add-on comparison file: {ADD_COMPARE_PATH}")
print(f"Raw new repositories present in add-on but not in base: {len(new_compare_rows)}")
print(f"New repositories inside the selected LOC range: {len(new_repo_names_in_loc_range)}")
new_repo_names_in_loc_range


Base comparison file: ../java-repos3-loc.json
Add-on comparison file: ../java-repos-add-loc.json
New repositories present in add-on but not in base: 10


['eugene-khyst/postgresql-event-sourcing',
 'facebook/SoLoader',
 'google/flogger',
 'licheedev/Android-SerialPort-API',
 'mauron85/react-native-background-geolocation',
 'mitre/HTTP-Proxy-Servlet',
 'ocpsoft/prettytime',
 'spring-guides/gs-rest-service',
 'square/seismic',
 'stealthcopter/AndroidNetworkTools']

In [8]:
new_repos_export_path = ADD_COMPARE_PATH.with_name(
    f"{ADD_COMPARE_PATH.stem.replace('-loc', '')}-only-new-in-loc-range-vs-{BASE_COMPARE_PATH.stem.replace('-loc', '')}.csv"
)

with new_repos_export_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    for repo_name in new_repo_names_in_loc_range:
        writer.writerow([repo_name])

print(f"Wrote {len(new_repo_names_in_loc_range)} new repository names to {new_repos_export_path}")


Wrote 10 new repository names to ../java-repos-add-only-new-vs-java-repos3.csv
